# 05C — Manual Validation and Final Sentiment Evaluation

This notebook compares **VADER** and the **Transformer** against human labels.

## Human labels allowed

- `Positive`
- `Neutral`
- `Negative`
- `Unclear`

`Unclear` rows are preserved but excluded from three-class evaluation.

Input:

`data/results/sentiment/manual_validation_sample_with_transformer.csv`

Main outputs:

- `manual_validation_reviewed.csv`
- `manual_method_evaluation.csv`
- confusion matrices
- method error files
- final method recommendation

> The notebook never invents human labels. When the `manual_label` column is
> empty, it reports progress and waits for manual annotation.


## 1. Imports and paths

Install the evaluation package when needed:

```bash
pip install scikit-learn
```

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    precision_recall_fscore_support,
)


def find_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("Project root not found.")


PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "sentiment"
FIGURE_DIR = PROJECT_ROOT / "figures" / "sentiment"

PRIMARY_SAMPLE = RESULTS_DIR / "manual_validation_sample_with_transformer.csv"
FALLBACK_SAMPLE = RESULTS_DIR / "manual_validation_sample.csv"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Primary sample:", PRIMARY_SAMPLE)


## 2. Load the annotation sample

In [ ]:
if PRIMARY_SAMPLE.exists():
    sample_path = PRIMARY_SAMPLE
elif FALLBACK_SAMPLE.exists():
    sample_path = FALLBACK_SAMPLE
else:
    raise FileNotFoundError(
        "No manual-validation sample was found. "
        "Run the VADER and Transformer notebooks first."
    )

manual_df = pd.read_csv(sample_path)

required = ["record_id", "text_clean_basic", "manual_label"]
missing = [column for column in required if column not in manual_df.columns]

if missing:
    raise ValueError("Missing columns: " + ", ".join(missing))

print("Loaded:", sample_path)
print("Rows:", len(manual_df))
print("VADER available:", "vader_label" in manual_df.columns)
print("Transformer available:", "transformer_label" in manual_df.columns)

display(manual_df.head(10))


## 3. Complete the manual labels

Open the CSV file and fill `manual_label` independently.

### Positive
Benefit, satisfaction, approval, optimism, or a clearly positive experience.

### Negative
Frustration, failure, concern, distrust, fear, or a clearly negative experience.

### Neutral
Mainly factual, descriptive, technical, or no clear opinion.

### Unclear
Ambiguous, mixed, mostly code, missing context, or difficult sarcasm.

Do not copy the automatic model label. The human label must be independent.


## 4. Normalize labels and show annotation progress

In [ ]:
NORMALIZATION = {
    "positive": "Positive",
    "pos": "Positive",
    "neutral": "Neutral",
    "neu": "Neutral",
    "negative": "Negative",
    "neg": "Negative",
    "unclear": "Unclear",
    "ambiguous": "Unclear",
    "mixed": "Unclear",
}

VALID = ["Positive", "Neutral", "Negative", "Unclear"]
EVALUATION_LABELS = ["Negative", "Neutral", "Positive"]


def normalize_label(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    return NORMALIZATION.get(text.lower(), text)


manual_df["manual_label"] = manual_df["manual_label"].map(normalize_label)

invalid = sorted(
    manual_df.loc[
        ~manual_df["manual_label"].isin(VALID + [""]),
        "manual_label",
    ].unique().tolist()
)

if invalid:
    raise ValueError(
        "Invalid labels found: "
        + ", ".join(invalid)
        + ". Use Positive, Neutral, Negative, or Unclear."
    )

progress = (
    manual_df["manual_label"]
    .replace("", np.nan)
    .fillna("Not annotated")
    .value_counts()
    .rename_axis("manual_label")
    .reset_index(name="records")
)
progress["percentage"] = (progress["records"] / len(manual_df) * 100).round(2)

display(progress)

annotated_count = int(manual_df["manual_label"].isin(VALID).sum())
evaluable_count = int(
    manual_df["manual_label"].isin(EVALUATION_LABELS).sum()
)
unclear_count = int(manual_df["manual_label"].eq("Unclear").sum())

print(f"Annotated: {annotated_count}/{len(manual_df)}")
print("Evaluable:", evaluable_count)
print("Unclear:", unclear_count)

reviewed_path = RESULTS_DIR / "manual_validation_reviewed.csv"
manual_df.to_csv(reviewed_path, index=False)
print("Saved:", reviewed_path)


## 5. Create the evaluable subset

In [ ]:
evaluation_df = manual_df.loc[
    manual_df["manual_label"].isin(EVALUATION_LABELS)
].copy()

print("Evaluable rows:", len(evaluation_df))

if evaluation_df.empty:
    print(
        "No Positive, Neutral, or Negative human labels are available yet. "
        "Fill the manual_label column, save the CSV, and rerun this notebook."
    )
else:
    display(
        evaluation_df[
            [
                column for column in [
                    "record_id",
                    "platform",
                    "text_clean_basic",
                    "manual_label",
                    "vader_label",
                    "transformer_label",
                ]
                if column in evaluation_df.columns
            ]
        ].head(10)
    )


## 6. Human-label distribution

Accuracy alone may be misleading when one class dominates. For this reason,
the notebook also calculates **macro F1**, which gives equal importance to all
three sentiment classes.


In [ ]:
if not evaluation_df.empty:
    human_distribution = (
        evaluation_df["manual_label"]
        .value_counts()
        .reindex(EVALUATION_LABELS, fill_value=0)
        .rename_axis("manual_label")
        .reset_index(name="records")
    )
    human_distribution["percentage"] = (
        human_distribution["records"] / len(evaluation_df) * 100
    ).round(2)

    display(human_distribution)
    human_distribution.to_csv(
        RESULTS_DIR / "manual_label_distribution.csv",
        index=False,
    )

    plt.figure(figsize=(8, 5))
    plt.bar(
        human_distribution["manual_label"],
        human_distribution["records"],
    )
    plt.title("Distribution of Human-Assigned Sentiment Labels")
    plt.xlabel("Human label")
    plt.ylabel("Number of records")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "manual_label_distribution.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## 7. Evaluation function

In [ ]:
def evaluate_method(data, prediction_column, method_name):
    valid = data[prediction_column].isin(EVALUATION_LABELS)
    y_true = data.loc[valid, "manual_label"]
    y_pred = data.loc[valid, prediction_column]

    if len(y_true) == 0:
        return None, None, None

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=EVALUATION_LABELS,
            average="macro",
            zero_division=0,
        )
    )

    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=EVALUATION_LABELS,
        average="weighted",
        zero_division=0,
    )

    metrics = {
        "method": method_name,
        "evaluated_records": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "cohen_kappa": cohen_kappa_score(
            y_true,
            y_pred,
            labels=EVALUATION_LABELS,
        ),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=EVALUATION_LABELS,
        output_dict=True,
        zero_division=0,
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=EVALUATION_LABELS,
    )

    return metrics, report, matrix


## 8. Evaluate VADER and Transformer

In [ ]:
evaluations = {}
metric_rows = []

for method_name, prediction_column in [
    ("VADER", "vader_label"),
    ("Transformer", "transformer_label"),
]:
    if (
        not evaluation_df.empty
        and prediction_column in evaluation_df.columns
    ):
        metrics, report, matrix = evaluate_method(
            evaluation_df,
            prediction_column,
            method_name,
        )

        if metrics is not None:
            evaluations[method_name] = {
                "metrics": metrics,
                "report": report,
                "matrix": matrix,
                "prediction_column": prediction_column,
            }
            metric_rows.append(metrics)

            print("\n", method_name)
            display(pd.DataFrame([metrics]).round(4))

            report_df = pd.DataFrame(report).transpose().round(4)
            display(report_df)

            report_df.to_csv(
                RESULTS_DIR
                / f"{method_name.lower()}_manual_classification_report.csv"
            )

if metric_rows:
    comparison_df = (
        pd.DataFrame(metric_rows)
        .round(4)
        .sort_values("macro_f1", ascending=False)
    )
    display(comparison_df)
    comparison_df.to_csv(
        RESULTS_DIR / "manual_method_evaluation.csv",
        index=False,
    )
else:
    comparison_df = pd.DataFrame()
    print("No evaluation was calculated yet.")


## 9. Confusion matrices

In [ ]:
for method_name, result in evaluations.items():
    matrix = result["matrix"]

    plt.figure(figsize=(7, 6))
    plt.imshow(matrix, aspect="auto")
    plt.xticks(range(3), EVALUATION_LABELS)
    plt.yticks(range(3), EVALUATION_LABELS)
    plt.xlabel(f"{method_name} prediction")
    plt.ylabel("Human label")
    plt.title(f"{method_name} Confusion Matrix")

    for row in range(3):
        for column in range(3):
            plt.text(
                column,
                row,
                matrix[row, column],
                ha="center",
                va="center",
            )

    plt.colorbar(label="Number of records")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / f"{method_name.lower()}_manual_confusion_matrix.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## 10. Compare method metrics

In [ ]:
if not comparison_df.empty:
    metric_columns = [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
    ]

    chart_data = (
        comparison_df
        .set_index("method")[metric_columns]
        .transpose()
    )

    plt.figure(figsize=(10, 6))
    chart_data.plot(kind="bar", ax=plt.gca())
    plt.title("Manual Evaluation of Sentiment Methods")
    plt.xlabel("Evaluation metric")
    plt.ylabel("Score")
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    plt.legend(title="Method")
    plt.tight_layout()
    plt.savefig(
        FIGURE_DIR / "manual_method_comparison.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## 11. Inspect model errors

The error files help identify sarcasm, technical language, mixed sentiment,
negation, and unclear context.


In [ ]:
error_columns = [
    column for column in [
        "record_id",
        "platform",
        "text_clean_basic",
        "manual_label",
        "vader_label",
        "vader_compound",
        "transformer_label",
        "transformer_confidence",
        "manual_notes",
    ]
    if column in evaluation_df.columns
]

for method_name, result in evaluations.items():
    prediction_column = result["prediction_column"]

    errors = evaluation_df.loc[
        evaluation_df[prediction_column]
        != evaluation_df["manual_label"],
        error_columns,
    ]

    print(f"{method_name} errors:", len(errors))
    display(errors.head(30))

    errors.to_csv(
        RESULTS_DIR / f"{method_name.lower()}_manual_errors.csv",
        index=False,
    )


## 12. Cases where one or both methods are wrong

In [ ]:
if (
    not evaluation_df.empty
    and "vader_label" in evaluation_df.columns
    and "transformer_label" in evaluation_df.columns
):
    both_wrong = evaluation_df.loc[
        (evaluation_df["vader_label"] != evaluation_df["manual_label"])
        & (
            evaluation_df["transformer_label"]
            != evaluation_df["manual_label"]
        ),
        error_columns,
    ]

    vader_only_correct = evaluation_df.loc[
        (evaluation_df["vader_label"] == evaluation_df["manual_label"])
        & (
            evaluation_df["transformer_label"]
            != evaluation_df["manual_label"]
        ),
        error_columns,
    ]

    transformer_only_correct = evaluation_df.loc[
        (
            evaluation_df["transformer_label"]
            == evaluation_df["manual_label"]
        )
        & (evaluation_df["vader_label"] != evaluation_df["manual_label"]),
        error_columns,
    ]

    print("Both wrong:", len(both_wrong))
    print("Only VADER correct:", len(vader_only_correct))
    print("Only Transformer correct:", len(transformer_only_correct))

    both_wrong.to_csv(
        RESULTS_DIR / "both_methods_wrong.csv",
        index=False,
    )
    vader_only_correct.to_csv(
        RESULTS_DIR / "vader_only_correct.csv",
        index=False,
    )
    transformer_only_correct.to_csv(
        RESULTS_DIR / "transformer_only_correct.csv",
        index=False,
    )

    display(both_wrong.head(20))
    display(vader_only_correct.head(20))
    display(transformer_only_correct.head(20))


## 13. Performance by platform

In [ ]:
platform_rows = []

if not evaluation_df.empty and "platform" in evaluation_df.columns:
    for platform, group in evaluation_df.groupby("platform"):
        row = {
            "platform": platform,
            "manual_records": len(group),
        }

        if "vader_label" in group.columns:
            row["vader_accuracy"] = (
                group["vader_label"] == group["manual_label"]
            ).mean()

        if "transformer_label" in group.columns:
            row["transformer_accuracy"] = (
                group["transformer_label"] == group["manual_label"]
            ).mean()

        platform_rows.append(row)

platform_evaluation = pd.DataFrame(platform_rows)

if not platform_evaluation.empty:
    display(platform_evaluation.round(4))
    platform_evaluation.to_csv(
        RESULTS_DIR / "manual_evaluation_by_platform.csv",
        index=False,
    )


## 14. Suggested primary method

The automatic recommendation is based mainly on **macro F1**.

It is only a suggestion. The final choice must also consider class-level
errors, platform behavior, and qualitative examples.


In [ ]:
if len(comparison_df) >= 2:
    best = comparison_df.sort_values(
        ["macro_f1", "accuracy"],
        ascending=False,
    ).iloc[0]

    recommendation = pd.DataFrame([
        {
            "recommended_primary_method": best["method"],
            "macro_f1": best["macro_f1"],
            "accuracy": best["accuracy"],
            "decision_basis": (
                "Higher macro F1 on the manually annotated sample"
            ),
            "warning": (
                "Review class-level and qualitative errors before final use"
            ),
        }
    ])

    display(recommendation)
    recommendation.to_csv(
        RESULTS_DIR / "sentiment_method_recommendation.csv",
        index=False,
    )
else:
    print(
        "A recommendation requires manual labels and evaluation "
        "for both VADER and the Transformer."
    )


## 15. Interpretation and limitations

Safe wording:

> On the manually annotated sample, one method achieved a higher macro
> F1-score than the other.

Avoid:

> The model perfectly understands all developer sentiment.

Important limitations:

- small manual sample;
- human subjectivity;
- class imbalance;
- unclear and mixed texts;
- technical developer language;
- platform differences;
- long texts truncated by the Transformer;
- model predictions are not ground truth.


## 16. Completion checklist

This phase is complete when:

- enough rows have human labels;
- invalid labels are absent;
- unclear cases are documented;
- VADER and Transformer metrics are calculated;
- confusion matrices are reviewed;
- error examples are inspected;
- one method is selected cautiously;
- evaluation files are saved.

The next technical phase is:

> **06 — Topic Modeling**


In [ ]:
print("Manual-validation notebook finished.")
print(f"Sample rows: {len(manual_df)}")
print(f"Annotated rows: {annotated_count}")
print(f"Evaluable rows: {evaluable_count}")
print(f"Unclear rows: {unclear_count}")

if evaluation_df.empty:
    print(
        "\nNext action: open the manual-validation CSV, "
        "fill manual_label, save it, and rerun this notebook."
    )
else:
    print(
        f"\nEvaluation used {len(evaluation_df)} human-labeled rows."
    )
